# Μάθημα 12 - Μείωση Ιστορικού Συνομιλίας με Agent Scratchpad

Αυτό το σημειωματάριο δείχνει πώς να διαχειρίζεστε το πλαίσιο σε μεγάλες συνομιλίες χρησιμοποιώντας το Microsoft Agent Framework. Καθώς οι συνομιλίες μεγαλώνουν, ο αριθμός των tokens αυξάνεται — τελικά ξεπερνώντας το παράθυρο συμφραζομένων του μοντέλου. Αντιμετωπίζουμε αυτό το θέμα με ένα **πρότυπο σύνοψης συμφραζομένων** και ένα **agent scratchpad** για επίμονη μνήμη.

## Τι θα μάθετε:
1. **Γιατί η Διαχείριση Συμφραζομένων είναι Σημαντική**: Κατανόηση των ορίων tokens και των παραθύρων συμφραζομένων
2. **Πράκτορες με Ενημερωμένη Συμφραζόμενη Πληροφορία**: Δημιουργία πρακτόρων που διαχειρίζονται το δικό τους πλαίσιο συνομιλίας
3. **Πρότυπο Σύνοψης Συμφραζομένων**: Χρήση εργαλείων για συμπύκνωση του ιστορικού συνομιλίας
4. **Agent Scratchpad**: Επίμονη μνήμη που επιβιώνει από τη μείωση του πλαισίου

## Προαπαιτούμενα:
- Ρύθμιση Azure OpenAI με παραμέτρους περιβάλλοντος διαμορφωμένες
- Κατανόηση βασικών εννοιών πρακτόρων από προηγούμενα μαθήματα


## Ρύθμιση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Γιατί η Διαχείριση του Πλαισίου Είναι Σημαντική

Κάθε LLM έχει ένα πεπερασμένο **παράθυρο πλαισίου** — τον μέγιστο αριθμό των tokens που μπορεί να επεξεργαστεί σε ένα μόνο αίτημα. Καθώς εξελίσσεται μια συνομιλία πολλαπλών γύρων:

- Ο **αριθμός tokens αυξάνεται γραμμικά** με κάθε μήνυμα χρήστη και απάντηση βοηθού.
- Οι **tokens του prompt κυριαρχούν στο κόστος** επειδή όλη η ιστορία αποστέλλεται ξανά σε κάθε γύρο.
- Τελικά η συνομιλία **υπερβαίνει το παράθυρο πλαισίου** και το μοντέλο είτε αποκόβει είτε εμφανίζει σφάλμα.

### Στρατηγικές για τη Διαχείριση του Πλαισίου

| Στρατηγική | Πώς Λειτουργεί | Ανταλλαγή |
|---|---|---|
| **Αποκοπή** | Απορρίπτει τα παλαιότερα μηνύματα | Χάνει το αρχικό πλαίσιο |
| **Περίληψη** | Συμπυκνώνει παλαιότερα μηνύματα σε μία περίληψη | Χάνεται κάποια λεπτομέρεια, αλλά διατηρούνται τα βασικά σημεία |
| **Scratchpad / Εξωτερική Μνήμη** | Αποθηκεύει βασικά στοιχεία εκτός της συνομιλίας | Απαιτεί κλήσεις εργαλείων, αλλά επιβιώνει οποιασδήποτε μείωσης |

Σε αυτό το τετράδιο συνδυάζουμε την **περίληψη** με ένα **εργαλείο scratchpad** ώστε ο πράκτορας να μπορεί να διατηρεί τη συνέχεια ακόμα και όταν η ιστορία της συνομιλίας συμπυκνώνεται.


## Δημιουργία ενός Πράκτορα με Επίγνωση Πλαισίου


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Προσομοίωση μιας Μακράς Συζήτησης

Ας κάνουμε μια σειρά από γύρους συνομιλίας για να δούμε πώς συσσωρεύεται το πλαίσιο. Ο πράκτορας θα πρέπει να διατηρήσει βασικές λεπτομέρειες (προτιμήσεις, προϋπολογισμό, ημερομηνίες ταξιδιού) κατά τη διάρκεια των γύρων και να δείξει συνέχεια.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Παρατηρήστε πώς ο πράκτορας διατηρεί το πλαίσιο από τις προηγούμενες ανταλλαγές — γνωρίζει για την Ιαπωνία, το σούσι, τους ναούς, τη φωτογραφία, τον προϋπολογισμό των $3000, το σόλο ταξίδι και το χρονικό διάστημα του Απριλίου. Σε μια σύντομη συνομιλία αυτό λειτουργεί καλά, αλλά καθώς η συνομιλία μεγαλώνει, το πλήρες ιστορικό γίνεται δαπανηρό να ξανασταλεί.

Ας συνεχίσουμε τη συνομιλία με περισσότερες ανταλλαγές για να δούμε τη συσσώρευση του πλαισίου:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Πρότυπο Συνοπτικής Ανακεφαλαίωσης Συμφραζομένων

Καθώς η συζήτηση εξελίσσεται, μπορούμε να χρησιμοποιήσουμε ένα **εργαλείο συνοπτικής ανακεφαλαίωσης** για να συμπτύξουμε τα συσσωρευμένα συμφραζόμενα σε μια συμπαγή μορφή. Ο πράκτορας καλεί αυτό το εργαλείο για να καταγράψει βασικές προτιμήσεις ώστε, ακόμη κι αν αφαιρεθούν παλαιότερα μηνύματα, οι βασικές πληροφορίες να διατηρηθούν.

Αυτό το πρότυπο είναι το θεμέλιο για πιο εξελιγμένη μείωση ιστορικού:
1. Ο πράκτορας εντοπίζει βασικά στοιχεία από τη συζήτηση
2. Καλεί το εργαλείο συνοπτικής ανακεφαλαίωσης για να τα αποθηκεύσει
3. Παλαιότερα μηνύματα μπορούν να αφαιρεθούν με ασφάλεια γιατί η σύνοψη καταγράφει ό,τι έχει σημασία

Παρακάτω ορίζουμε ένα εργαλείο `summarize_preferences` το οποίο μπορεί να καλεί ο πράκτορας για να καταγράψει μια συμπαγή σύνοψη όσων έχει μάθει.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να διαχειρίζεστε το πλαίσιο σε συνομιλίες αντιπροσώπων που διαρκούν πολύ χρησιμοποιώντας το Microsoft Agent Framework:

### Βασικές Έννοιες
- **Τα παράθυρα πλαισίου είναι πεπερασμένα** — κάθε token στο ιστορικό συνομιλίας κοστίζει χρήματα και μετράει προς το όριο.
- **Εργαλεία συνοπτικής παρουσίασης** επιτρέπουν στον αντιπρόσωπο να συμπυκνώνει το συσσωρευμένο πλαίσιο σε συμπαγείς συνοπτικές περιλήψεις, μειώνοντας τη χρήση tokens ενώ διατηρείται η βασική πληροφορία.
- **Σημειωματάρια αντιπροσώπου** παρέχουν επίμονη εξωτερική μνήμη που διατηρείται ανεξαρτήτως περικοπής της συνομιλίας.

### Τι Δημιουργήσατε
- Έναν **αντιπρόσωπο που αναγνωρίζει το πλαίσιο** και διατηρεί τη συνέχεια σε συνομιλίες πολλών γύρων
- Ένα **εργαλείο συνοπτικής παρουσίασης** (`summarize_preferences`) που καταγράφει βασικές λεπτομέρειες χρήστη σε συμπαγή μορφή
- Μια **πολυστροφή συνομιλία** που δείχνει τη διατήρηση του πλαισίου και την αντιμετώπιση αλλαγών

### Εφαρμογές στον Πραγματικό Κόσμο
- **Ρομπότ Εξυπηρέτησης Πελατών**: Θυμούνται προτιμήσεις κατά τη διάρκεια μεγάλων συνεδριών υποστήριξης
- **Προσωπικοί Βοηθοί**: Παρακολουθούν συνεχιζόμενα έργα χωρίς να απαιτείται επανεξήγηση του πλαισίου
- **Εκπαιδευτικοί Δάσκαλοι**: Διατηρούν την πρόοδο του μαθητή σε πολλές αλληλεπιδράσεις

### Επόμενα Βήματα
- Υλοποίηση πλήρους εργαλείου σημειωματαρίου με επιμονή βάσει αρχείων
- Προσθήκη αυτόματης περικοπής ιστορικού μετά τη συνοπτική παρουσίαση
- Συνδυασμός με βάσεις δεδομένων διάνυσμα για αναζήτηση σημασιολογικής μνήμης
- Δημιουργία αντιπροσώπων που μπορούν να συνεχίσουν συνομιλίες ημέρες αργότερα με πλήρες πλαίσιο


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθύνης**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μηχανικής μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε σημειώστε ότι οι αυτόματες μεταφράσεις ενδέχεται να περιέχουν σφάλματα ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η επίσημη πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική μετάφραση από ανθρώπους. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
